# 03 — Clustering on Gene Indicator Features

Adapted from [SalvatoreRa/tutorial — hierarchical_cluster_and_K_means](https://github.com/SalvatoreRa/tutorial/blob/main/genomic%20series/hierarchical_cluster_and_K_means.ipynb) and [DBSCAN_and_GMM](https://github.com/SalvatoreRa/tutorial/blob/main/genomic%20series/DBSCAN_and_GMM.ipynb) (Apache-2.0).

**Purpose:** Identify patient subgroups via unsupervised clustering on `G__*` gene indicators from the merged GENIE dataset. Compare K-Means, hierarchical, DBSCAN, and Gaussian mixture approaches.

**Inputs:**
- `data/features/pca_gene_features.csv` (from NB02) or raw gene columns from source data

**Outputs:**
- Dendrograms, silhouette plots, cluster scatter → `reports/figures/`
- Cluster assignments → `data/features/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
FEAT_DIR     = os.path.join(PROJECT_ROOT, "data", "features")
FIG_DIR      = os.path.join(PROJECT_ROOT, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ── Load PCA features (from NB02) or raw gene columns ────────────────
pca_path = os.path.join(FEAT_DIR, "pca_gene_features.csv")
if os.path.exists(pca_path):
    X = pd.read_csv(pca_path, index_col=0)
    print(f"Loaded PCA features: {X.shape}")
else:
    # Fallback: load raw gene columns from source
    DATA_PATH = os.path.join(PROJECT_ROOT, "datasets_analysis_dictionary", "merged_genie.xlsx")
    df = pd.read_excel(DATA_PATH)
    gene_cols = [c for c in df.columns if c.startswith("G__")]
    X = df[gene_cols].dropna()
    X = pd.DataFrame(StandardScaler().fit_transform(X), columns=gene_cols, index=X.index)
    print(f"Loaded raw gene features: {X.shape}")

In [ ]:
# ── Elbow method + silhouette for K-Means ────────────────────────────
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, "o-", color="steelblue")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(K_range, sil_scores, "o-", color="darkorange")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "clustering_elbow_silhouette.png"), dpi=150, bbox_inches="tight")
plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k}")

In [ ]:
# ── Hierarchical clustering dendrogram ───────────────────────────────
Z = linkage(X.values, method="ward")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax,
           leaf_rotation=90, leaf_font_size=9)
ax.set_title("Hierarchical Clustering Dendrogram (Ward)")
ax.set_xlabel("Sample index (or cluster size)")
ax.set_ylabel("Distance")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "clustering_dendrogram.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Compare methods at best_k ────────────────────────────────────────
methods = {
    "KMeans":         KMeans(n_clusters=best_k, n_init=10, random_state=42),
    "Agglomerative":  AgglomerativeClustering(n_clusters=best_k),
    "GMM":            GaussianMixture(n_components=best_k, random_state=42),
    "DBSCAN":         DBSCAN(eps=1.5, min_samples=5),
}

fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 5))
results = {}

for ax, (name, model) in zip(axes, methods.items()):
    if name == "GMM":
        labels = model.fit_predict(X)
    else:
        labels = model.fit_predict(X)
    results[name] = labels
    n_clusters = len(set(labels) - {-1})
    sil = silhouette_score(X, labels) if n_clusters > 1 else float("nan")
    ax.scatter(X.iloc[:, 0], X.iloc[:, 1], c=labels, cmap="tab10", alpha=0.5, s=10)
    ax.set_title(f"{name}\nk={n_clusters}  sil={sil:.3f}")
    ax.set_xlabel(X.columns[0])
    ax.set_ylabel(X.columns[1])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "clustering_method_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Export cluster assignments ────────────────────────────────────────
cluster_df = pd.DataFrame(results, index=X.index)
cluster_df.to_csv(os.path.join(FEAT_DIR, "cluster_assignments.csv"))
print(f"Saved cluster assignments → data/features/cluster_assignments.csv")